In [1]:
  import pandas as pd 
  import numpy as np
  import matplotlib.pyplot as plt
  import warnings
  warnings.filterwarnings('ignore')
  
  from sklearn.model_selection import train_test_split
  from sklearn.preprocessing import StandardScaler
  from sklearn.linear_model import LogisticRegression
  from sklearn.metrics import roc_auc_score, RocCurveDisplay, PrecisionRecallDisplay
  from xgboost import XGBClassifier
  import pickle

In [2]:
  df = pd.read_csv('../data/processed/customer_features.csv')

  FEATURES = [
      'recency_days', 'frequency', 'monetary', 'avg_order_value',
      'avg_review_score', 'min_review_score', 'review_count',
      'num_unique_products', 'category_diversity', 'avg_freight_ratio',
      'credit_card_ratio', 'max_installments',
      'avg_delivery_delay_days', 'late_delivery_count'
  ]

  X = df[FEATURES]
  y = df['churned']   
  
  X_train, X_test, y_train, y_test = train_test_split(
      X, y, test_size=0.2, random_state=42, stratify=y)

  print(f'Train: {len(X_train):,} 样本')
  print(f'Test:  {len(X_test):,} 样本')
  print(f'Train → Retained: {(y_train==0).sum()} | Churned: {(y_train==1).sum()}')

Train: 44,213 样本
Test:  11,054 样本
Train → Retained: 520 | Churned: 43693


In [3]:
  scaler = StandardScaler()
  X_train_sc = scaler.fit_transform(X_train)
  X_test_sc  = scaler.transform(X_test)

  lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
  lr.fit(X_train_sc, y_train)   

  lr_auc = roc_auc_score(y_test, lr.predict_proba(X_test_sc)[:, 1])
  print(f'Logistic Regression AUC: {lr_auc:.4f}')



Logistic Regression AUC: 0.5751


In [4]:
  # 计算 scale_pos_weight 处理类别不平衡
  n_pos = (y_train == 1).sum()   # churned
  n_neg = (y_train == 0).sum()   # retained
  spw   = n_neg / n_pos
  print(f'Retained: {n_neg} | Churned: {n_pos}')
  print(f'scale_pos_weight = {spw:.4f}')

  xgb = XGBClassifier(
      n_estimators  = 300,
      max_depth     = 4,
      learning_rate = 0.05,
      subsample     = 0.8,
      colsample_bytree = 0.8,   
      #scale_pos_weight = spw,
      eval_metric   = 'auc',
      random_state  = 42,
      verbosity     = 0
  )

  xgb.fit(
      X_train, y_train,
      eval_set        = [(X_test, y_test)],
      verbose         = 50
  )

  xgb_auc = roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1])
  print(f'\nXGBoost AUC: {xgb_auc:.4f}')

Retained: 520 | Churned: 43693
scale_pos_weight = 0.0119
[0]	validation_0-auc:0.59638
[50]	validation_0-auc:0.59267
[100]	validation_0-auc:0.59911
[150]	validation_0-auc:0.58710
[200]	validation_0-auc:0.58192
[250]	validation_0-auc:0.58076
[299]	validation_0-auc:0.57691

XGBoost AUC: 0.5769


In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/olist.db')

In [9]:
  review_query = """
  SELECT 
      c.customer_unique_id,
      GROUP_CONCAT(r.review_comment_message, ' ') as review_text
  FROM orders o
  JOIN customers c ON o.customer_id = c.customer_id
  LEFT JOIN order_reviews r ON o.order_id = r.order_id
  WHERE o.order_status = 'delivered'
    AND o.order_purchase_timestamp >= '2017-01-01'
    AND o.order_purchase_timestamp <  '2018-03-01'
    AND r.review_comment_message IS NOT NULL
  GROUP BY c.customer_unique_id
  """

  reviews = pd.read_sql_query(review_query, conn)
  conn.close()

  print(f'有评论文本的客户数: {len(reviews)}')
  print(f'示例评论（前3条）:')
  print(reviews['review_text'].head(3).values)

有评论文本的客户数: 22860
示例评论（前3条）:
['Bom vendedor'
 'Olá! Comprei dois potes de whey e chegou apenas um! Não consigo rastrear o outro, pois os dois constam no mesmo número do pedido. E a nota fiscal emitida também consta como sendo dos dois produtos. '
 'só achei que a altura da saia poderia ser maior']


In [10]:
  from sklearn.feature_extraction.text import TfidfVectorizer
  from sklearn.decomposition import TruncatedSVD
  
  # 合并评论文本到主特征表（无评论的填空字符串）
  df3 = df.merge(reviews, on='customer_unique_id', how='left')
  df3['review_text'] = df3['review_text'].fillna('')

  print(f'有评论文本的客户比例: {(df3["review_text"] != "").mean():.1%}')
  
  # TF-IDF（葡萄牙语，限制词汇量）
  tfidf = TfidfVectorizer(
      max_features = 200,
      min_df       = 3,
      ngram_range  = (1, 2),
      sublinear_tf = True
  )
  tfidf_matrix = tfidf.fit_transform(df3['review_text'])

  # SVD 降到 15 维（避免维度灾难） 
  svd = TruncatedSVD(n_components=15, random_state=42)
  tfidf_svd = svd.fit_transform(tfidf_matrix)
  print(f'TF-IDF SVD 解释方差: {svd.explained_variance_ratio_.sum():.1%}')

  # 添加进特征表
  for i in range(15): 
      df3[f'review_topic_{i}'] = tfidf_svd[:, i]

  FEATURES_V3 = FEATURES + [f'review_topic_{i}' for i in range(15)]
  print(f'总特征数: {len(FEATURES_V3)}')
  


有评论文本的客户比例: 41.4%
TF-IDF SVD 解释方差: 34.2%
总特征数: 29


In [16]:
  from sklearn.model_selection import train_test_split
  from sklearn.metrics import roc_auc_score
  from xgboost import XGBClassifier

  X3 = df3[FEATURES_V3]
  y3 = df3['churned'] 

  X_train3, X_test3, y_train3, y_test3 = train_test_split(
      X3, y3, test_size=0.2, random_state=42, stratify=y3)

  xgb_final = XGBClassifier(
      n_estimators          = 400,
      max_depth             = 4,
      learning_rate         = 0.05,
      subsample             = 0.8,
      colsample_bytree      = 0.7,
      eval_metric           = 'auc',
      random_state          = 42,
      verbosity             = 0,
      early_stopping_rounds = 20      # 连续20步不提升就停止
  )
  
  xgb_final.fit(
      X_train3, y_train3,
      eval_set = [(X_test3, y_test3)],
      verbose  = 20   
  )

  auc_final = roc_auc_score(y_test3, xgb_final.predict_proba(X_test3)[:,1])
  print(f'\n最终 AUC: {auc_final:.4f}')
  print(f'最优迭代轮次: {xgb_final.best_iteration}')

[0]	validation_0-auc:0.55027
[20]	validation_0-auc:0.60300
[30]	validation_0-auc:0.59372

最终 AUC: 0.6117
最优迭代轮次: 10


In [17]:


  with open('../data/processed/xgb_model.pkl', 'wb') as f:
      pickle.dump(xgb_final, f) 

  df3['churn_prob'] = xgb_final.predict_proba(df3[FEATURES_V3])[:, 1]
  df3.to_csv('../data/processed/customer_features_scored.csv', index=False)

  X_test3.to_csv('../data/processed/X_test.csv', index=False)
  y_test3.to_csv('../data/processed/y_test.csv', index=False)

  # 保存特征名列表（SHAP 需要用）
  import json
  with open('../data/processed/feature_names.json', 'w') as f:
      json.dump(FEATURES_V3, f)

  print('✓ 模型已保存')
  print('✓ 打分数据已保存')
  print(f'churn_prob 分布：\n{df3["churn_prob"].describe().round(4)}')

✓ 模型已保存
✓ 打分数据已保存
churn_prob 分布：
count    55267.0000
mean         0.9235
std          0.0036
min          0.7492
25%          0.9232
50%          0.9241
75%          0.9243
max          0.9245
Name: churn_prob, dtype: float64
